# Scaling and OptimizationLearn how to scale the ADEPT framework for production workloads.## Topics Covered- Horizontal scaling strategies- Database optimization- Caching with Redis- Load balancing- Performance tuning

In [ ]:
# Check current resource usage
import psutil
import docker

client = docker.from_env()

print("Container Resource Usage:")
print("-" * 60)

for container in client.containers.list():
    stats = container.stats(stream=False)
    cpu_delta = stats['cpu_stats']['cpu_usage']['total_usage'] - \
                stats['precpu_stats']['cpu_usage']['total_usage']
    system_delta = stats['cpu_stats']['system_cpu_usage'] - \
                   stats['precpu_stats']['system_cpu_usage']
    cpu_percent = (cpu_delta / system_delta) * 100 if system_delta > 0 else 0
    
    memory_usage = stats['memory_stats']['usage'] / (1024 * 1024)  # MB
    memory_limit = stats['memory_stats']['limit'] / (1024 * 1024)  # MB
    
    print(f"{container.name:25} CPU: {cpu_percent:.1f}% | "
          f"Memory: {memory_usage:.0f}/{memory_limit:.0f} MB")

## Auto-scaling Configuration

In [ ]:
# HPA configuration example
hpa_config = '''
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: agent-gateway-hpa
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: agent-gateway
  minReplicas: 2
  maxReplicas: 10
  metrics:
  - type: Resource
    resource:
      name: cpu
      target:
        type: Utilization
        averageUtilization: 70
  - type: Resource
    resource:
      name: memory
      target:
        type: Utilization
        averageUtilization: 80
'''

print(hpa_config)